In [1]:
import pickle

with open(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/token_id_to_genename_all.pkl",
    "rb",
) as f:
    token_dict = pickle.load(f)

print(f"Vocab size: {len(token_dict)}")
print(f"Suggested TGT_VOCAB_SIZE: {len(token_dict)}")

Vocab size: 1466
Suggested TGT_VOCAB_SIZE: 1466


In [2]:
from datasets import load_from_disk

ds = load_from_disk(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/dataset_all_src/obese.dataset"
)
print(ds)
print(f"Max sequence length: {max(len(x) for x in ds['input_ids'])}")

/rds/general/user/ap5625/home/miniforge3/envs/perturbgen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['input_ids', 'cell_states_adipocytes', 'condition', 'cell_pairing_index', 'length'],
    num_rows: 1083
})
Max sequence length: 344


In [1]:
import scanpy as sc

src = sc.read_h5ad(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_all_src/obese.h5ad"
)
tgt = sc.read_h5ad(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_all_tgt/1_weightloss.h5ad"
)

print(f"Src genes: {src.n_vars}")
print(f"Tgt genes: {tgt.n_vars}")

Src genes: 1974
Tgt genes: 1974


/rds/general/user/ap5625/home/miniforge3/envs/perturbgen/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [2]:
# check what the decoder outputs (to see if i need to re run the training on just the tokenised counts(
import torch

ckpt = torch.load(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/res/count/checkpoints/20260609_1327_cellgen_train_count_lr_0.001_wd_0.001_batch_16_drop_0.1_zinb_tp_1_s_0_pos_time_pos_sin_m_cosine-epoch=14.ckpt",
    map_location="cpu",
    weights_only=False,
)

# Find the count decoder output layer
for k, v in ckpt["state_dict"].items():
    if "count" in k.lower() and hasattr(v, "shape"):
        print(f"{k}: {v.shape}")

decoder.count_decoder.mlp.fc1.weight: torch.Size([768, 768])
decoder.count_decoder.mlp.fc1.bias: torch.Size([768])
decoder.count_decoder.mlp.fc2.weight: torch.Size([768, 768])
decoder.count_decoder.mlp.fc2.bias: torch.Size([768])
decoder.count_decoder.linear_output.weight: torch.Size([1974, 768])
decoder.count_decoder.linear_output.bias: torch.Size([1974])
decoder.count_decoder.softmax_output.0.weight: torch.Size([1974, 768])
decoder.count_decoder.softmax_output.0.bias: torch.Size([1974])


In [3]:
import pickle

with open(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/tokenid_to_rowid_all.pkl",
    "rb",
) as f:
    rowid = pickle.load(f)
print(f"tokenid_to_rowid entries: {len(rowid)}")
print(f"Max row_id: {max(rowid.values())}")

tokenid_to_rowid entries: 1466
Max row_id: 1465
